<a href="https://colab.research.google.com/github/zhengyangsu/Natural-Language-Processing/blob/main/latin_translation_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latin to English Translation with LoRA

Adapted from the Week 1 tutorial `movie_review_sentiment_classifier_with_bert.ipynb`  
Use a GPU runtime

# Setup Python Libraries (pip)

In [ ]:
# transformers = models and tokenizers, peft = LoRA, bitsandbytes = 4-bit loading,
# accelerate = GPU placement, datasets = data loading, --quiet hides the long install log
!pip install transformers peft bitsandbytes accelerate datasets

# Load the Dataset

In [ ]:
from datasets import load_dataset  #downloads datasets from Hugging Face

raw = load_dataset('grosenthal/latin_english_parallel')  # download the Latin-English pairs (train / test / valid splits) https://huggingface.co/datasets/grosenthal/latin_english_parallel

dataset = {  # a plain dict holding the three small subsets we will use
    'train': raw['train'].shuffle(seed=42).select(range(2000)),  # shuffle with a fixed seed, keep 2000 rows for training
    'val':   raw['valid'].select(range(200)),   # first 200 validation rows: used to monitor loss during training
    'test':  raw['test'].select(range(200)),    # first 200 test rows: used only for the final before/after comparison
}

#import json
'''
Final test data JSONL format
pairs = [
    {"la": "Arma virumque cano.", "en": "I sing of arms and the man."},
    {"la": "Carpe diem.", "en": "Seize the day."},
]'''

# Tokenize the Dataset

In [ ]:
from transformers import AutoTokenizer  # class that loads the matching tokenizer for a model

checkpoint = 'hathibelagal/llama-3.2-latin'  #base model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)  #download the tokenizer that matches the model's vocabulary
tokenizer.pad_token = tokenizer.eos_token  # Llama has no padding token, so reuse the end-of-text token

def tokenization(batch):  # function applied to a batch of rows
    # join each Latin sentence with its English translation into one prompt-style text
    texts = [f"Latin: {la}\nEnglish: {en}" for la, en in zip(batch['la'], batch['en'])]
    return tokenizer(texts, truncation=True, max_length=256)  # text -> token ids; cap to 256 tokens

tokenized = {k: v.map(tokenization, batched=True, remove_columns=v.column_names)  # run it on each split in batches, drop raw text columns
             for k, v in dataset.items()}  # loop over train / val / test

# Load the Model and Add LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True,  # store the frozen base weights in 4 bits
                         bnb_4bit_quant_type='nf4',  # NF4 is the 4-bit format designed for QLoRA
                         bnb_4bit_compute_dtype=torch.float16)  # compute in 16-bit floats

model = AutoModelForCausalLM.from_pretrained(checkpoint,  # download base model
                                             quantization_config=bnb,  # apply the 4-bit settings above
                                             device_map='auto')  # training on the GPU automatically

model = prepare_model_for_kbit_training(model)  # casts some layers to full precision, cant update 4bit weight

model = get_peft_model(model, LoraConfig(r=16,  # rank of the LoRA matrices: higher = more capacity
                                         lora_alpha=32,  # scaling factor for the LoRA update
                                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # add LoRA to the attention layers
                                         task_type='CAUSAL_LM'))  # word-prediction with Parameter Efficient Fine Tuning（
model.print_trainable_parameters()  # show how few parameters are actually trained

# Setup Training Configurations

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling  # training tools

training_args = TrainingArguments(
    seed=42,  # fixed random seed for reproducibility
    output_dir='./results',  # folder for checkpoints
    num_train_epochs=1,  # one full pass over the training data
    per_device_train_batch_size=4,  # 4 examples per step (small to fit GPU memory)
    gradient_accumulation_steps=4,  # accumulate 4 steps before each update = effective batch size 16
    learning_rate=2e-4,  # step size
    fp16=True,  # 16-bit training (T4 supports fp16 not bf16)
    eval_strategy='epoch',  # evaluate on the validation set at the end of each epoch
    report_to='none',  # do not log
)

trainer = Trainer(
    model=model,  # the LoRA model to train
    args=training_args,  # hyperparameter settings above
    train_dataset=tokenized['train'],  # data to learn from
    eval_dataset=tokenized['val'],  # data used for evaluation during training
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),  # builds padded batches and labels; mlm=False = predict the next word
)

# Evaluate the Un-Fine-Tuned Model for a Baseline

In [ ]:
import numpy as np  # for exp()

result = trainer.evaluate(tokenized['test'])  # loss on the test set (a fresh LoRA starts at zero = the un-fine-tuned model)
print(result, 'perplexity =', np.exp(result['eval_loss']))  # perplexity = exp(loss): lower = less "surprised" by the text

# Fine-Tune with LoRA

In [ ]:
trainer.train()  # run the LoRA fine-tuning loop

# Evaluate After Fine-Tuning

In [ ]:
result = trainer.evaluate(tokenized['test'])  # same test set as the baseline, so the numbers are comparable
print(result, 'perplexity =', np.exp(result['eval_loss']))  # compare with the baseline perplexity

# Try out some examples and save the adapter

In [ ]:
def translate(la):  # translate one Latin sentence
    inputs = tokenizer(f"Latin: {la}\nEnglish:", return_tensors='pt').to(model.device)  # build prompt, make tensors, move to GPU
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)  # generate up to 100 new tokens, greedy (no randomness)
    text = tokenizer.decode(out[0], skip_special_tokens=True)  # token ids back to text
    return text.split('English:')[-1].split('\n')[0].strip()  # keep only the first line after "English:" and trim spaces

print(translate("Quo usque tandem abutere, Catilina, patientia nostra?"))  # try a famous Cicero line

model.save_pretrained('latin_lora')  # save only the small LoRA adapter, not the 3B base model